## Einführung 

Diese Lektion behandelt: 
- Was Funktionsaufrufe sind und wofür sie verwendet werden 
- Wie man einen Funktionsaufruf mit Azure OpenAI erstellt 
- Wie man einen Funktionsaufruf in eine Anwendung integriert 

## Lernziele 

Nach Abschluss dieser Lektion wissen Sie, wie man und verstehen: 

- Den Zweck der Verwendung von Funktionsaufrufen 
- Einrichtung von Funktionsaufrufen mit dem Azure Open AI-Dienst 
- Entwurf effektiver Funktionsaufrufe für den Anwendungsfall Ihrer Anwendung 


## Verständnis von Funktionsaufrufen 

Für diese Lektion möchten wir ein Feature für unser Bildungs-Startup entwickeln, das es den Nutzern ermöglicht, einen Chatbot zu verwenden, um technische Kurse zu finden. Wir werden Kurse empfehlen, die zu ihrem Fähigkeitsniveau, ihrer aktuellen Rolle und der Technologie von Interesse passen. 

Um dies zu erreichen, verwenden wir eine Kombination aus: 
 - `Azure Open AI`, um eine Chat-Erfahrung für den Nutzer zu schaffen
 - `Microsoft Learn Catalog API`, um Nutzern zu helfen, Kurse basierend auf ihrer Anfrage zu finden 
 - `Function Calling`, um die Anfrage des Nutzers aufzunehmen und an eine Funktion weiterzuleiten, die die API-Anfrage stellt. 

Um zu beginnen, schauen wir uns an, warum wir Funktionsaufrufe überhaupt verwenden möchten: 


### Warum Funktionsaufrufe  

Wenn Sie eine andere Lektion in diesem Kurs abgeschlossen haben, verstehen Sie wahrscheinlich die Leistungsfähigkeit der Verwendung von Large Language Models (LLMs). Hoffentlich können Sie auch einige ihrer Einschränkungen erkennen. 

Funktionsaufrufe sind ein Feature des Azure Open AI Service, um die folgenden Einschränkungen zu überwinden: 
1) Konsistentes Antwortformat 
2) Möglichkeit, Daten aus anderen Quellen einer Anwendung in einem Chat-Kontext zu nutzen 

Vor Funktionsaufrufen waren die Antworten eines LLM unstrukturiert und inkonsistent. Entwickler mussten komplexen Validierungscode schreiben, um sicherzustellen, dass sie jede Variation einer Antwort verarbeiten können. 

Benutzer konnten keine Antworten wie „Wie ist das aktuelle Wetter in Stockholm?“ erhalten. Das liegt daran, dass Modelle auf den Zeitpunkt des Trainingsdaten begrenzt waren. 

Schauen wir uns das folgende Beispiel an, das dieses Problem veranschaulicht: 

Nehmen wir an, wir wollen eine Datenbank mit Studentendaten erstellen, um ihnen den passenden Kurs vorzuschlagen. Unten haben wir zwei Beschreibungen von Studenten, die in den Daten sehr ähnlich sind.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Wir möchten dies an ein LLM senden, um die Daten zu analysieren. Dies kann später in unserer Anwendung verwendet werden, um es an eine API zu senden oder in einer Datenbank zu speichern. 

Lassen Sie uns zwei identische Aufforderungen erstellen, in denen wir das LLM anweisen, welche Informationen uns interessieren: 


Wir wollen dies an ein LLM senden, um die für unser Produkt wichtigen Teile zu analysieren. So können wir zwei identische Eingabeaufforderungen erstellen, um das LLM anzuweisen: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Nachdem wir diese beiden Eingabeaufforderungen erstellt haben, senden wir sie mit `client.responses.create` an das LLM. Wir speichern die Eingabeaufforderung in der Variablen `input` und weisen die Rolle `user` zu. Dies soll eine Nachricht von einem Benutzer simulieren, die an einen Chatbot geschrieben wird. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Jetzt können wir beide Anfragen an das LLM senden und die erhaltene Antwort überprüfen. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Obwohl die Eingabeaufforderungen gleich sind und die Beschreibungen ähnlich sind, können wir unterschiedliche Formate der Eigenschaft `Grades` erhalten.

Wenn Sie die obige Zelle mehrmals ausführen, kann das Format `3.7` oder `3.7 GPA` sein.

Das liegt daran, dass das LLM unstrukturierte Daten in Form der geschriebenen Eingabeaufforderung aufnimmt und ebenfalls unstrukturierte Daten zurückgibt. Wir benötigen ein strukturiertes Format, damit wir wissen, was zu erwarten ist, wenn wir diese Daten speichern oder verwenden.

Durch die Verwendung von Funktionsaufrufen können wir sicherstellen, dass wir strukturierte Daten zurückerhalten. Beim Funktionsaufruf ruft das LLM tatsächlich keine Funktionen auf oder führt sie aus. Stattdessen erstellen wir eine Struktur, der das LLM für seine Antworten folgen soll. Dann verwenden wir diese strukturierten Antworten, um zu wissen, welche Funktion wir in unseren Anwendungen ausführen müssen.
 


![Funktionsaufrufflussdiagramm](../../../../translated_images/de/Function-Flow.083875364af4f4bb.webp)


Wir können dann das, was von der Funktion zurückgegeben wird, nehmen und an das LLM zurücksenden. Das LLM wird dann in natürlicher Sprache antworten, um die Anfrage des Benutzers zu beantworten. 


### Anwendungsfälle für die Verwendung von Funktionsaufrufen 

**Externe Tools aufrufen** 
Chatbots sind hervorragend darin, Nutzern auf Fragen Antworten zu geben. Durch die Verwendung von Funktionsaufrufen können die Chatbots Nachrichten von Nutzern nutzen, um bestimmte Aufgaben zu erledigen. Zum Beispiel kann ein Schüler den Chatbot bitten, "Sende eine E-Mail an meinen Dozenten, in der steht, dass ich mehr Unterstützung bei diesem Thema benötige". Dies kann einen Funktionsaufruf an `send_email(to: string, body: string)` auslösen.


**API- oder Datenbankabfragen erstellen**
Nutzer können Informationen in natürlicher Sprache anfragen, die dann in eine formatierte Abfrage oder API-Anfrage umgewandelt wird. Ein Beispiel wäre ein Lehrer, der fragt: "Wer sind die Schüler, die die letzte Aufgabe abgeschlossen haben?", was eine Funktion namens `get_completed(student_name: string, assignment: int, current_status: string)` aufrufen könnte.


**Strukturierte Daten erstellen**
Nutzer können einen Textblock oder eine CSV-Datei verwenden und das LLM dazu nutzen, wichtige Informationen daraus zu extrahieren. Zum Beispiel kann ein Schüler einen Wikipedia-Artikel über Friedensabkommen umwandeln, um KI-Lernkarten zu erstellen. Dies kann durch die Verwendung einer Funktion namens `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` geschehen.


## 2. Erstellen Ihres ersten Funktionsaufrufs 

Der Prozess zur Erstellung eines Funktionsaufrufs umfasst 3 Hauptschritte: 
1. Aufruf der Chat Completions API mit einer Liste Ihrer Funktionen und einer Benutzer-Nachricht 
2. Lesen der Antwort des Modells, um eine Aktion auszuführen, z. B. eine Funktion oder API-Aufruf ausführen 
3. Erneuter Aufruf der Chat Completions API mit der Antwort Ihrer Funktion, um diese Informationen zu verwenden, um eine Antwort an den Benutzer zu erstellen. 


![Ablauf eines Funktionsaufrufs](../../../../translated_images/de/LLM-Flow.3285ed8caf4796d7.webp)


### Elemente eines Funktionsaufrufs 

#### Benutzereingabe 

Der erste Schritt besteht darin, eine Benutzernachricht zu erstellen. Diese kann dynamisch zugewiesen werden, indem der Wert einer Texteingabe übernommen wird, oder Sie können hier einen Wert zuweisen. Wenn Sie zum ersten Mal mit der Chat Completions API arbeiten, müssen wir die `role` und den `content` der Nachricht definieren. 

Die `role` kann entweder `system` (Regeln erstellen), `assistant` (das Modell) oder `user` (der Endbenutzer) sein. Für den Funktionsaufruf weisen wir dies als `user` und eine Beispiel-Frage zu. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Funktionen erstellen. 

Als Nächstes definieren wir eine Funktion und die Parameter dieser Funktion. Wir verwenden hier nur eine Funktion namens `search_courses`, aber Sie können mehrere Funktionen erstellen.

**Wichtig** : Funktionen sind in der Systemnachricht an das LLM enthalten und werden in die Anzahl der verfügbaren Tokens eingerechnet.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definitionen** 

`name` - Der Name der Funktion, die aufgerufen werden soll. 

`description` - Dies ist die Beschreibung, wie die Funktion funktioniert. Hier ist es wichtig, spezifisch und klar zu sein 

`parameters` - Eine Liste von Werten und Formaten, die das Modell in seiner Antwort erzeugen soll 


`type` - Der Datentyp, in dem die Eigenschaften gespeichert werden 

`properties` - Liste der spezifischen Werte, die das Modell für seine Antwort verwendet 


`name` - der Name der Eigenschaft, die das Modell in seiner formatierten Antwort verwendet 

`type` - Der Datentyp dieser Eigenschaft 

`description` - Beschreibung der spezifischen Eigenschaft 


**Optional**

`required` - erforderliche Eigenschaft, damit der Funktionsaufruf abgeschlossen wird 


### Den Funktionsaufruf durchführen
Nachdem wir eine Funktion definiert haben, müssen wir sie nun in den Aufruf der Chat Completion API einfügen. Dies tun wir, indem wir `functions` zur Anfrage hinzufügen. In diesem Fall `functions=functions`.

Es gibt auch die Möglichkeit, `function_call` auf `auto` zu setzen. Das bedeutet, dass wir das LLM entscheiden lassen, welche Funktion basierend auf der Benutzernachricht aufgerufen werden soll, anstatt sie selbst zuzuweisen.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Schauen wir uns nun die Antwort an und sehen, wie sie formatiert ist: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Sie können sehen, dass der Name der Funktion aufgerufen wird und basierend auf der Benutzernachricht das LLM die Daten gefunden hat, um die Argumente der Funktion zu erfüllen. 


## 3. Integration von Funktionsaufrufen in eine Anwendung. 


Nachdem wir die formatierte Antwort des LLM getestet haben, können wir diese nun in eine Anwendung integrieren. 

### Steuerung des Ablaufs 

Um dies in unsere Anwendung zu integrieren, gehen wir wie folgt vor: 

Zuerst rufen wir die Open AI-Dienste auf und speichern die Nachricht in einer Variable namens `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Nun definieren wir die Funktion, die die Microsoft Learn API aufruft, um eine Liste von Kursen zu erhalten: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Als bewährte Praxis werden wir dann prüfen, ob das Modell eine Funktion aufrufen möchte. Danach erstellen wir eine der verfügbaren Funktionen und ordnen sie der Funktion zu, die aufgerufen wird. 
Anschließend nehmen wir die Argumente der Funktion und ordnen sie den Argumenten des LLM zu.

Schließlich fügen wir die Funktionsaufrufnachricht und die Werte hinzu, die von der `search_courses`-Nachricht zurückgegeben wurden. Dies gibt dem LLM alle Informationen, die es benötigt, um
mit natürlicher Sprache auf den Benutzer zu antworten. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Nun werden wir die aktualisierte Nachricht an das LLM senden, damit wir eine Antwort in natürlicher Sprache erhalten, anstatt eine API-JSON-formatierte Antwort. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Code Challenge 

Tolle Arbeit! Um dein Lernen mit Azure Open AI Function Calling fortzusetzen, kannst du folgendes erstellen: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Weitere Parameter der Funktion, die Lernenden helfen könnten, mehr Kurse zu finden. Die verfügbaren API-Parameter findest du hier: 
 - Erstelle einen weiteren Funktionsaufruf, der mehr Informationen vom Lernenden abfragt, wie z.B. deren Muttersprache 
 - Erstelle eine Fehlerbehandlung, falls der Funktionsaufruf und/oder API-Aufruf keine passenden Kurse zurückgibt 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
